# Bimodality analysis: does the reliability-mixture model reproduce the two-peak phenomenon?

Runs on Kaggle or Colab. Needs two inputs attached:
1. `data01_direction4priors.csv` (same raw data used throughout this project)
2. The 12 fitted `subject_N.pkl` checkpoints from `kaggle_hb_multisubject_fit.ipynb`
   (and its follow-up), attached the same way checkpoints were carried between
   fitting chunks earlier: publish that notebook's output as a dataset, then
   Add Input here.

**This notebook is the actual analysis, not a plotting script.** Every cell
from Section 2 onward computes something (the per-trial predicted
distribution, whether it's bimodal, how that varies by condition). The plots
at the end visualize real numbers this notebook itself calculated, they
don't re-derive anything separately.


In [ ]:
import glob
import pickle
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import i0e
from scipy.signal import find_peaks

print("imports ok")


## 1. Load data and checkpoints

Same pattern as the fitting notebooks: prefer an uploaded input over cloning,
so this works with or without internet access.


In [ ]:
candidates = glob.glob("/kaggle/input/**/data01_direction4priors.csv", recursive=True)
candidates += glob.glob("./**/data01_direction4priors.csv", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "data01_direction4priors.csv not found. Attach it via Add Input > Upload."
    )
CSV_PATH = candidates[0]
print(f"Found data at: {CSV_PATH}")

checkpoint_paths = sorted(glob.glob("/kaggle/input/**/subject_*.pkl", recursive=True))
checkpoint_paths += sorted(glob.glob("./**/subject_*.pkl", recursive=True))
checkpoints = {}
for p in checkpoint_paths:
    subject_id = int(p.split("subject_")[-1].split(".")[0].split("_")[0])
    checkpoints[subject_id] = p
checkpoints = dict(sorted(checkpoints.items()))
print(f"Found {len(checkpoints)} checkpoints: {list(checkpoints.keys())}")
if len(checkpoints) != 12:
    print("WARNING: expected 12 checkpoints. Attach all subject_N.pkl files as inputs.")


## 2. The model itself

Same reliability-mixture model used for fitting (see
`reliability_mixture_model.py` in the repo for the fully documented,
importable version). Inlined here so this notebook is self-contained on
Kaggle/Colab without needing a separate module file attached.


In [ ]:
DEG = np.arange(1, 361)


def load_and_prepare_data(csv_path):
    raw = pd.read_csv(csv_path)
    data = raw.copy()
    data['estimate_deg'] = np.degrees(np.arctan2(data.estimate_y, data.estimate_x)) % 360
    data.loc[data.estimate_deg == 0, 'estimate_deg'] = 360
    data = data.dropna(subset=['estimate_deg', 'motion_direction']).copy()
    data['error_deg'] = ((data.estimate_deg - data.motion_direction + 180) % 360) - 180
    data['block_id'] = data['subject_id'].astype(str) + '_' + data['run_id'].astype(str)

    trial_data = data.rename(columns={
        'subject_id': 'subject', 'trial_index': 'trial', 'prior_std': 'prior_width',
        'motion_coherence': 'coherence', 'motion_direction': 'true_direction',
    })[['subject', 'block_id', 'trial', 'prior_width', 'coherence',
        'true_direction', 'estimate_deg', 'error_deg', 'session_id', 'experiment_id']]

    block_context = (
        data.groupby(['subject_id', 'run_id'], as_index=False)
        .agg(session_id=('session_id', 'first'))
        .sort_values(['subject_id', 'run_id']).reset_index(drop=True)
    )
    block_context['prev_session_id'] = block_context.groupby('subject_id')['session_id'].shift(1)
    block_context['same_session_prev'] = (
        block_context['prev_session_id'].notna()
        & (block_context['session_id'] == block_context['prev_session_id'])
    )
    block_context['block_id'] = block_context['subject_id'].astype(str) + '_' + block_context['run_id'].astype(str)
    trial_data = trial_data.merge(
        block_context[['block_id', 'run_id', 'same_session_prev']], on='block_id', how='left'
    )
    return trial_data


def vm_pdf(support_deg, mu_deg, k, norm=True):
    mu = np.atleast_1d(np.asarray(mu_deg, float))
    x = np.deg2rad(np.asarray(support_deg, float))[:, None]
    u = np.deg2rad(mu)[None, :]
    k = float(k)
    if np.isinf(k) or k > 1e300:
        out = np.zeros((len(support_deg), len(mu)))
        for j, mm in enumerate(mu):
            out[np.argmin(np.abs(np.asarray(support_deg) - mm)), j] = 1.0
    else:
        out = np.exp(k * np.cos(x - u) - k) / (2 * np.pi * i0e(k))
    if norm:
        out = out / out.sum(0, keepdims=True)
    return out


def wrap_signed_deg(a):
    return ((np.asarray(a) + 180) % 360) - 180


def circular_mean_deg(angles_deg):
    r = np.deg2rad(np.asarray(angles_deg))
    return float(np.degrees(np.arctan2(np.mean(np.sin(r)), np.mean(np.cos(r)))) % 360)


def prior_agreement(measurement_deg, prior_mean_deg, k_prior):
    d_ = np.deg2rad(wrap_signed_deg(measurement_deg - prior_mean_deg))
    return float(np.exp(k_prior * (np.cos(d_) - 1.0)))


def trial_percept_distribution(prior_mean_deg, measurement_deg, k_prior, k_llh, prior_reliance):
    prior_component = vm_pdf(DEG, prior_mean_deg, k_prior)[:, 0]
    llh_component = vm_pdf(DEG, measurement_deg, k_llh)[:, 0]
    mixture = prior_reliance * prior_component + (1 - prior_reliance) * llh_component
    return mixture / mixture.sum()


def circ_convolve_vec(p, kernel):
    return np.real(np.fft.ifft(np.fft.fft(p) * np.fft.fft(kernel)))


## 3. The calculation: predicted distribution and bimodality, every trial, every subject

For each subject, replays the same sequential trial loop used during fitting
(so `prior_reliance` evolves exactly as it did when these parameters were
fit), and for every trial:

1. Computes the full predicted response distribution (360 bins)
2. Checks whether that distribution has two or more separated peaks (tiled
   circular peak-finding, 30-degree minimum separation, matching the
   methodology used to compare against the GPU project's own multimodality
   screen)
3. Opportunistically keeps the *first* clearly bimodal trial and the *first*
   clearly unimodal (narrow-prior) trial encountered per subject, for the
   plots in Section 5, these are real trials from this same pass, not
   separately searched for afterward.


In [ ]:
N_ANGLES = 360
MIN_SEPARATION_DEG = 30.0
MIN_BINS = max(1, round(MIN_SEPARATION_DEG / (360.0 / N_ANGLES)))
PROMINENCE_RATIOS = (0.05, 0.10)


def has_separated_peaks(peaks, n_bins, minimum_bins):
    for i, first in enumerate(peaks):
        for second in peaks[i + 1:]:
            distance = abs(int(first) - int(second))
            if min(distance, n_bins - distance) >= minimum_bins:
                return True
    return False


def is_multimodal(pmf, prominence_ratio, minimum_bins=MIN_BINS):
    extended = np.tile(pmf, 3)
    peaks, _ = find_peaks(extended, prominence=prominence_ratio * pmf.max(), distance=minimum_bins)
    central = peaks[(peaks >= N_ANGLES) & (peaks < 2 * N_ANGLES)] - N_ANGLES
    return len(central) >= 2 and has_separated_peaks(central, N_ANGLES, minimum_bins)


trial_data = load_and_prepare_data(CSV_PATH)

trial_records = []       # one row per subject x prior_width x coherence
subject_records = []     # one row per subject x prominence threshold
example_trials = {}      # captured during the same pass, used for plotting below

for subject_id, checkpoint_path in checkpoints.items():
    with open(checkpoint_path, "rb") as f:
        checkpoint = pickle.load(f)
    params = checkpoint["best_params"]
    k_llh, k_prior = params["k_llh"], params["k_prior"]
    alpha, k_motor, lapse_rate = params["alpha"], params["k_motor"], params["lapse_rate"]

    d = trial_data[trial_data.subject == subject_id].sort_values(["run_id", "trial"]).reset_index(drop=True)
    motor_kernel = vm_pdf(DEG, 360.0, k_motor)[:, 0]

    reliance = 0.5
    window_buf = deque(maxlen=5)
    prev_block_id = None

    block_ids = d["block_id"].to_numpy()
    same_sess = d["same_session_prev"].to_numpy()
    true_dirs = d["true_direction"].to_numpy(dtype=float)
    coherences = d["coherence"].to_numpy()
    prior_widths = d["prior_width"].to_numpy()

    multimodal_flags = {r: np.empty(len(d), dtype=bool) for r in PROMINENCE_RATIOS}
    condition_flags = []  # (prior_width, coherence, multimodal_at_5pct)

    for i in range(len(d)):
        is_new_block = block_ids[i] != prev_block_id
        if is_new_block and not bool(same_sess[i]):
            window_buf.clear()

        kp = k_prior[prior_widths[i]]
        kl = k_llh[coherences[i]]
        mixture = trial_percept_distribution(225.0, true_dirs[i], kp, kl, reliance)
        with_motor = circ_convolve_vec(mixture, motor_kernel)
        with_motor = np.clip(with_motor, 0, None)
        with_motor /= with_motor.sum()
        pmf = (1 - lapse_rate) * with_motor + lapse_rate * (1.0 / 360)

        for r in PROMINENCE_RATIOS:
            multimodal_flags[r][i] = is_multimodal(pmf, r)
        this_bimodal = bool(multimodal_flags[0.05][i])
        condition_flags.append((prior_widths[i], coherences[i], this_bimodal))

        # capture real examples for the plot, opportunistically, from this same pass
        if subject_id not in example_trials:
            example_trials[subject_id] = {}
        ex = example_trials[subject_id]
        if "bimodal_wide" not in ex and prior_widths[i] == 80 and this_bimodal:
            ex["bimodal_wide"] = dict(pmf=pmf.copy(), reliance=reliance,
                                       true_direction=true_dirs[i], prior_width=80, coherence=coherences[i])
        if "unimodal_narrow" not in ex and prior_widths[i] == 10 and not this_bimodal:
            ex["unimodal_narrow"] = dict(pmf=pmf.copy(), reliance=reliance,
                                          true_direction=true_dirs[i], prior_width=10, coherence=coherences[i])

        window_buf.append(true_dirs[i])
        smoothed = circular_mean_deg(list(window_buf))
        agreement = prior_agreement(smoothed, 225.0, kp)
        reliance = float(np.clip(reliance + alpha * (agreement - reliance), 1e-4, 1 - 1e-4))
        prev_block_id = block_ids[i]

    for r in PROMINENCE_RATIOS:
        subject_records.append(dict(
            subject=subject_id, prominence_fraction_of_maximum=r,
            candidate_multimodal_trials=int(multimodal_flags[r].sum()),
            total_trials=len(d),
            candidate_multimodal_fraction=float(multimodal_flags[r].mean()),
        ))

    cond_df = pd.DataFrame(condition_flags, columns=["prior_width", "coherence", "multimodal"])
    cond_summary = cond_df.groupby(["prior_width", "coherence"])["multimodal"].mean().reset_index()
    cond_summary["subject"] = subject_id
    trial_records.append(cond_summary)

    print(f"subject {subject_id}: multimodal fraction @5% prominence = "
          f"{multimodal_flags[0.05].mean():.3f}, @10% = {multimodal_flags[0.10].mean():.3f}")

summary_df = pd.DataFrame(subject_records)
condition_df = pd.concat(trial_records, ignore_index=True)
summary_df.to_csv("our_model_multimodality_screen.csv", index=False)
condition_df.to_csv("our_model_multimodality_by_condition.csv", index=False)
print("\nSaved: our_model_multimodality_screen.csv, our_model_multimodality_by_condition.csv")


## 4. Summary tables

The actual result: how often the model's per-trial predictions are genuinely
bimodal, overall and broken down by condition.


In [ ]:
print("=== Per-subject summary ===")
display(summary_df)

print("\n=== By condition, averaged across subjects ===")
display(condition_df.pivot_table(index="prior_width", columns="coherence", values="multimodal", aggfunc="mean").round(3))

print("\n=== Overall ===")
display(summary_df.groupby("prominence_fraction_of_maximum")["candidate_multimodal_fraction"].agg(["mean", "min", "max"]))


## 5. Visualizing a few real trials from the calculation above

These are not searched for separately, `example_trials` was populated
during the Section 3 pass, the first bimodal wide-prior trial and first
unimodal narrow-prior trial actually encountered for each subject. This is
just a view into what Section 3 already computed.


In [ ]:
def plot_subject_examples(subject_id, ax_bimodal, ax_unimodal):
    ex = example_trials.get(subject_id, {})
    for ax, key, label in [(ax_bimodal, "bimodal_wide", "80\u00b0 prior"),
                            (ax_unimodal, "unimodal_narrow", "10\u00b0 prior")]:
        if key not in ex:
            ax.set_title(f"Subject {subject_id}: no {label} example found")
            continue
        e = ex[key]
        ax.plot(DEG, e["pmf"], color="#2c5f8a", linewidth=1.6)
        ax.fill_between(DEG, e["pmf"], alpha=0.15, color="#2c5f8a")
        ax.axvline(225, color="#c0392b", linestyle="--", linewidth=1.3, label="Prior mean (225\u00b0)")
        ax.axvline(e["true_direction"], color="#27ae60", linestyle="--", linewidth=1.3,
                   label=f"True direction ({e['true_direction']:.0f}\u00b0)")
        ax.set_title(f"Subject {subject_id}, {label}, {e['coherence']*100:.0f}% coherence\n"
                     f"prior_reliance = {e['reliance']:.2f}", fontsize=9.5)
        ax.set_xlabel("Response direction (degrees)")
        ax.set_ylabel("Predicted probability")
        ax.set_xlim(0, 360)
        ax.legend(fontsize=7.5, loc="upper right")
        ax.spines[["top", "right"]].set_visible(False)


# Compare a subject with a clean multimodality profile against one flagged
# in the k_prior-identifiability finding (edit these IDs to look at others)
subjects_to_plot = [s for s in [4, 5] if s in checkpoints]

fig, axes = plt.subplots(len(subjects_to_plot), 2, figsize=(11, 4.3 * len(subjects_to_plot)))
if len(subjects_to_plot) == 1:
    axes = axes.reshape(1, 2)
for row, subject_id in enumerate(subjects_to_plot):
    plot_subject_examples(subject_id, axes[row, 0], axes[row, 1])

fig.suptitle("Real predicted distributions from Section 3's calculation", fontsize=12.5, y=1.01)
fig.tight_layout()
fig.savefig("bimodality_example_trials.png", dpi=160, bbox_inches="tight")
plt.show()
print("saved bimodality_example_trials.png")
